# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Detección de Argumentos con Gemma 4B

- ollama serve
- ollama run gemma3:4b

In [1]:
#%pip install langchain pymupdf openai openpyxl --quiet



In [3]:
from typing import List
from pydantic import BaseModel, Field, ValidationError
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from langchain_core.exceptions import OutputParserException
import requests
import json
import re
from openai import OpenAI
import openai
import httpx
import pandas as pd
import numpy as np
import os
import openpyxl

process_text_path = "..\\Data\\Processed Files (sections)\\"

model_name="gemma3:4b"
prefix = 'GLOBAL_SGD2024_'
output_dir = "..\\Data\\Extracted Arguments No Keywords (all text)\\"

## Input text processing

In [2]:
# 1. Define your Pydantic schema for output
class ArgumentResponse(BaseModel):
    arguments: List[str] = Field(..., description="List of arguments extracted directly from the text.")

# 2. Setup output parser
pydantic_parser = PydanticOutputParser(pydantic_object=ArgumentResponse)

# 3. Extend text with first sentence from the next page
def extend_pages_with_next_sentence(pages):
    def get_first_sentence(text):
        match = re.search(r'(.+?\.)', text.strip())
        return match.group(1).strip() if match else ""

    extended_pages = []
    for i, page in enumerate(pages):
        current_text = page["text"]
        if i + 1 < len(pages):
            next_sentence = get_first_sentence(pages[i + 1]["text"])
            current_text += " " + next_sentence
        extended_pages.append({
            "page": page["page"],
            "text": current_text
        })
    return extended_pages

# 4. Build the prompt and call the LLM to extract arguments
def extract_arguments_json(text, topic, model_name) -> ArgumentResponse:
    format_instructions = pydantic_parser.get_format_instructions()

    prompt = PromptTemplate(
        template=(
            "Task: Text Span Identification for Arguments related to Sustainable Development Goal: {topic}\n\n"
            "Role: You are an expert in logical reasoning, sustainability reporting, and argument analysis. "
            "Your job is to identify and extract **verbatim arguments** about {topic} from long-form sustainability texts.\n\n"
            "Instructions:\n"
            "1. Carefully read the entire input text.\n"
            "2. Identify ONLY those sentences or phrases that:\n"
            "   - Clearly support or argue for or against the topic {topic}\n"
            "   - Contain keyword from the relevant lists below\n"
            "   - Are exclusively about {topic} (EXCLUDE if they mention or refer to other SDGs or unrelated sustainability topics)\n\n"
            "3. Each extracted argument must:\n"
            "   - Relate exclusively to the specified SDG ({topic})\n"
            "   - Stand as a full statement\n"
            "   - Be copied exactly from the original (no paraphrasing)\n"
            "   - Include only the necessary context for understanding\n"
            "4. If no qualifying arguments are found, return an empty array.\n\n"
            "Output Rules:\n"
            "- Use **only the exact text** from the original\n"
            "- Do **not** add or reword anything\n"
            "- Return only valid JSON\n"
            "- No markdown (```), no extra explanation\n\n"
            "Text:\n\"\"\"\n{text}\n\"\"\"\n\n"
            "Respond ONLY with a JSON object like this:\n\n"
            "{format_instructions}"
        ),
        input_variables=["text", "topic"],
        partial_variables={"format_instructions": format_instructions}
    )

    final_prompt = prompt.format_prompt(text=text, topic=topic).to_string()

    payload = {
        "model": model_name,
        "prompt": final_prompt,
        "temperature": 0,
        "stream": False
    }

    response = requests.post("http://localhost:11434/api/generate", json=payload)
    if response.status_code != 200:
        raise Exception(f"Ollama error: {response.text}")

    raw_output = response.json()["response"]
    print("Model Output:", raw_output)

    try:
        return pydantic_parser.parse(raw_output)
    except OutputParserException as err:
        print("Parse failed:", err)
        return ArgumentResponse(arguments=[])

# 5. Wrapper function for pipeline
def extract_arguments_from_text(text, topic, model_name) -> List[str]:
    result = extract_arguments_json(text, topic, model_name)
    return result.arguments

# 6. Main document-level processor
def process_document(pages, model_name, topic=""):
    extended_pages = extend_pages_with_next_sentence(pages)
    processed = []
    for page in extended_pages:
        print(f"\n--- Processing Page {page['page']} ---")
        #print("Text to analyze:\n", page["text"])
        
        arguments = extract_arguments_from_text(page["text"], topic, model_name)
        
        print("Extracted Arguments:")
        for i, arg in enumerate(arguments, 1):
            print(f"{i}. {arg}")

        processed.append({
            "page": page["page"],
            "text": page["text"],
            "arguments": arguments
        })
    return processed


# 7. File I/O
def save_to_json(processed, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(processed, f, indent=2, ensure_ascii=False)

def process_directory(input_dir, output_dir, prefix, model_name, topic="", sgd_number=None):
    os.makedirs(output_dir, exist_ok=True)
    all_results = []

    for filename in os.listdir(input_dir):
        if filename.endswith(".json") and filename.startswith(prefix):
            filepath = os.path.join(input_dir, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                pages = json.load(f)

            section_name = filename.replace(".json", "")
            processed = process_document(pages, model_name, topic)

            for item in processed:
                item["section"] = section_name  # Add section identifier
                all_results.append(item)
                
    return all_results



## SGD 1: Poverty

In [ ]:
topic = "SGD 1 (Poverty): End poverty in all its forms everywhere"
sgd_number = "1"
resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "Of the 169 targets, 135 can be assessed using available global trend data from the 2015 baseline to the most recent year, along with custodian agency analyses;",
    "only 17 per cent display progress sufficient for achievement by 2030.",
    "Nearly half exhibit moderate to severe deviations from the desired trajectory, with 30 per cent showing marginal progress and 18 per cent moderate progress.",
    "18 per cent indicate stagnation and 17 per cent regression below the 2015 baseline levels.",
    "only 17 per cent display progress sufficient for achievement by 2030."
  ]
}
```
Extracted Arguments:
1. Of the 169 targets, 135 can be assessed using available global trend data from the 2015 baseline to the most recent year, along with custodian agency analyses;
2. only 17 per cent display progress sufficient for achievement by 2030.
3. Nearly half exhibit moderate to severe deviations from the desired trajectory, w

## SGD 2: Hunger

In [ ]:
topic = "SGD 2 (Hunger): End hunger, achieve food security and improved nutrition and promote sustainable agriculture"
sgd_number = "2"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "The current status of the SDGs: severely off track",
    "critical gaps persist",
    "major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16)",
    "Approximately one third of indicators lack data for the past three years, hampering the ability of policymakers to make timely informed decisions and course corrections"
  ]
}
```
Extracted Arguments:
1. The current status of the SDGs: severely off track
2. critical gaps persist
3. major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16)
4. Approximately one third of indicators lack data for the past three years, hampering the ability of policymakers to make timely informed decisions and course corrections

--- Processing Page 7 ---
Model Output: ```json
{
  "ar

## SGD 3: Health

In [ ]:
topic = "SGD 3 (Health): Ensure healthy lives and promote well-being for all at all ages"
sgd_number = "3"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "Alarmingly, 18 per cent indicate stagnation and 17 per cent regression below the 2015 baseline levels.",
    "The variance in country data coverage plays out differently across the Goals, with major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16).",
    "Approximately one third of indicators lack data for the past three years, hampering the ability of policymakers to make timely informed decisions and course corrections.",
    "Overall progress across targets based on 2015–2024 global aggregate data"
  ]
}
```
Extracted Arguments:
1. Alarmingly, 18 per cent indicate stagnation and 17 per cent regression below the 2015 baseline levels.
2. The variance in country data coverage plays out differently across the Goals, with major shortfalls in priority development areas, such as gender equality (Goal 5), climate a

## SGD 4: Education

In [ ]:
topic = "SGD 4 (Education): Ensure inclusive and equitable quality education"
sgd_number = "4"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "critical gaps persist",
    "major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16)",
    "Approximately one third of indicators lack data for the past three years, hampering the ability of policymakers to make timely informed decisions and course corrections",
    "good data coverage"
  ]
}
```
Extracted Arguments:
1. critical gaps persist
2. major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16)
3. Approximately one third of indicators lack data for the past three years, hampering the ability of policymakers to make timely informed decisions and course corrections
4. good data coverage

--- Processing Page 7 ---
Model Output: ```json
{
  "arguments": [
    "Data demand for SDG monitoring has spurred inno

## SGD 5: Gender

In [ ]:
topic = "SGD 5 (Gender): Achieve gender equality and empower all women and girls"
sgd_number = "5"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "major shortfalls in priority development areas, such as gender equality (Goal 5)",
    "gender equality (Goal 5)"
  ]
}
```
Extracted Arguments:
1. major shortfalls in priority development areas, such as gender equality (Goal 5)
2. gender equality (Goal 5)

--- Processing Page 7 ---
Model Output: ```json
{
  "arguments": [
    "A comprehensive review by the Gender Data Compass, however, revealed significant gaps in disaggregated data for key social, economic and environmental indicators disseminated online by national statistical offices (NSOs).",
    "Sex-disaggregated data were most available for indicators on demographics, migration, fertility and reproductive health (with a score of 83) and least available for environmental indicators (23).",
    "High-income countries generally exhibited lower overall disaggregated data availability compared to low- and middle-income countries.",
    "A comprehensive review b

## SGD 6: Water and sanitation

In [ ]:
topic = "SGD 6 (Water and sanitation): Ensure availability and sustainable management of water and sanitation for all"
sgd_number = "6"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)




--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "SDG progress assessment is affected by ongoing challenges in the availability and timeliness of data to monitor all 17 Goals.",
    "critical gaps persist",
    "SDG 6 (Water and sanitation): Ensure availability and sustainable management of water and sanitation"
  ]
}
```
Extracted Arguments:
1. SDG progress assessment is affected by ongoing challenges in the availability and timeliness of data to monitor all 17 Goals.
2. critical gaps persist
3. SDG 6 (Water and sanitation): Ensure availability and sustainable management of water and sanitation

--- Processing Page 7 ---
Model Output: ```json
{
  "arguments": [
    "Population censuses provide crucial data for national and subnational planning and also serve as denominators for numerous SDG indicators.",
    "Data demand for SDG monitoring has spurred innovation and the integration of different data sources.",
    "Countries worldwide are recognizing the need to

## SGD 7: Clean Energy

In [ ]:
topic = "SGD 7 (Clean Energy): Ensure access to affordable, reliable, sustainable and modern energy for all"
sgd_number = "7"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "critical gaps persist",
    "SDG progress assessment is affected by ongoing challenges in the availability and timeliness of data to monitor all 17 Goals",
    "data timeliness remains a challenge",
    "Approximately one third of indicators lack data for the past three years, hampering the ability of policymakers to make timely informed decisions and course corrections",
    "major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16)"
  ]
}
```
Extracted Arguments:
1. critical gaps persist
2. SDG progress assessment is affected by ongoing challenges in the availability and timeliness of data to monitor all 17 Goals
3. data timeliness remains a challenge
4. Approximately one third of indicators lack data for the past three years, hampering the ability of policymakers to make timely informed decisions and course co

## SGD 8: Decent Work, Economic Growth

In [ ]:
topic = "SGD 8 (decent work, economic growth): Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all"
sgd_number = "8"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "Of the 169 targets, 135 can be assessed using available global trend data from the 2015 baseline to the most recent year, along with custodian agency analyses; 34 targets lack sufficient trend data or additional analysis.",
    "Nearly half (48 per cent) exhibit moderate to severe deviations from the desired trajectory, with 30 per cent showing marginal progress and 18 per cent moderate progress.",
    "Approximately one third of indicators lack data for the past three years, hampering the ability of policymakers to make timely informed decisions and course corrections.",
    "The variance in country data coverage plays out differently across the Goals, with major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16)."
  ]
}
```
Extracted Arguments:
1. Of the 169 targets, 135 can be assessed using available global 

## SGD 9: Infrastructure, industrilization, innovation

In [4]:
topic = "SGD 9 (Infrastructure, industrilization, innovation): Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation"
sgd_number = "9"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "Of the 169 targets, 135 can be assessed using available global trend data from the 2015 baseline to the most recent year, along with custodian agency analyses",
    "Only 17 per cent display progress sufficient for achievement by 2030",
    "Nearly half exhibit moderate to severe deviations from the desired trajectory, with 30 per cent showing marginal progress and 18 per cent moderate progress",
    "18 per cent indicate stagnation and 17 per cent regression below the 2015 baseline levels",
    "Good trend data are available for 51 per cent of the indicators (at least two data points since 2015) in more than half of all countries"
  ]
}
```
Extracted Arguments:
1. Of the 169 targets, 135 can be assessed using available global trend data from the 2015 baseline to the most recent year, along with custodian agency analyses
2. Only 17 per cent display progress sufficient for achievement by 2030
3. Nearly half exhibit

## SGD 10: Inequality

In [5]:
topic = "SGD 10 (Inequality): Reduce inequality within and among countries"
sgd_number = "10"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "critical gaps persist",
    "major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16)",
    "Approximately one third of indicators lack data for the past three years, hampering the ability of policymakers to make timely informed decisions and course corrections"
  ]
}
```
Extracted Arguments:
1. critical gaps persist
2. major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16)
3. Approximately one third of indicators lack data for the past three years, hampering the ability of policymakers to make timely informed decisions and course corrections

--- Processing Page 7 ---
Model Output: ```json
{
  "arguments": [
    "Data demand for SDG monitoring has spurred innovation and the integration of different data sou

## SGD 11: Sustainable cities

In [6]:
topic = "SGD 11 (Sustainable Cities, Sustainable Communities): Make cities and human settlements inclusive, safe, resilient and sustainable"
sgd_number = "11"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "critical gaps persist",
    "major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16)"
  ]
}
```
Extracted Arguments:
1. critical gaps persist
2. major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16)

--- Processing Page 7 ---
Model Output: ```json
{
  "arguments": [
    "Countries worldwide are recognizing the need to invest in national statistical systems to produce high-quality, timely data for SDG monitoring.",
    "Data demand for SDG monitoring has spurred innovation and the integration of different data sources.",
    "Both the Netherlands and Uruguay monitor air pollution with national networks of sensors (SDG indicator 11.6.2).",
    "The Gender Data Compass, however, revealed significant gaps in d

## SGD 12: Responsible Consumption, Responsible Production

In [7]:
topic = "SGD 12 (Responsible Consumption, Responsible Production): Ensure sustainable consumption and production patterns"
sgd_number = "12"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "critical gaps persist",
    "data were available for more than 50 per cent of countries",
    "data were available for more than 50 per cent of countries",
    "data were available for more than 50 per cent of countries",
    "data were available for more than 50 per cent of countries",
    "data were available for more than 50 per cent of countries",
    "data were available for more than 50 per cent of countries",
    "data were available for more than 50 per cent of countries"
  ]
}
```
Extracted Arguments:
1. critical gaps persist
2. data were available for more than 50 per cent of countries
3. data were available for more than 50 per cent of countries
4. data were available for more than 50 per cent of countries
5. data were available for more than 50 per cent of countries
6. data were available for more than 50 per cent of countries
7. data were available for more than 50 per cent of countries
8. data were a

## SGD 13: Climate change

In [8]:
topic = "SGD 13 (Climate change): Take urgent action to combat climate change and its impacts"
sgd_number = "13"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "critical gaps persist",
    "climate action (Goal 13)",
    "urgent need for intensified efforts to put the SDGs on course",
    "data timeliness remains a challenge",
    "climate action (Goal 13)"
  ]
}
```
Extracted Arguments:
1. critical gaps persist
2. climate action (Goal 13)
3. urgent need for intensified efforts to put the SDGs on course
4. data timeliness remains a challenge
5. climate action (Goal 13)

--- Processing Page 7 ---
Model Output: ```json
{
  "arguments": [
    "Countries worldwide are recognizing the need to invest in national statistical systems to produce high-quality, timely data for SDG monitoring.",
    "Data demand for SDG monitoring has spurred innovation and the integration of different data sources.",
    "Engaging citizens in data production is essential to leave no one behind.",
    "Data availability score for sex-disaggregated data ranged from 23 to 83 out of 100."
  ]
}
```
Extr

## SGD 14: Life bellow water

In [9]:
topic = "SGD 14 (Life bellow Water): Conserve and sustainably use the oceans, seas and marine resources for sustainable development"
sgd_number = "14"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "Among the assessable targets, only 17 per cent display progress sufficient for achievement by 2030.",
    "Nearly half (48 per cent) exhibit moderate to severe deviations from the desired trajectory, with 30 per cent showing marginal progress and 18 per cent moderate progress.",
    "Alarmingly, 18 per cent indicate stagnation and 17 per cent regression below the 2015 baseline levels.",
    "The variance in country data coverage plays out differently across the Goals, with major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16).",
    "Approximately one third of indicators lack data for the past three years, hampering the ability of policymakers to make timely informed decisions and course corrections.",
    "Good trend data are available for 51 per cent of the indicators (at least two data points since 2015) i

## SGD 15: Life on land

In [ ]:
topic = "SGD 15 (Life on land): Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss"
sgd_number = "15"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 6 ---
Model Output: ```json
{
  "arguments": [
    "critical gaps persist",
    "major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16)",
    "Approximately one third of indicators lack data for the past three years, hampering the ability of policymakers to make timely informed decisions and course corrections",
    "Good trend data are available for 51 per cent of the indicators (at least two data points since 2015) in more than half of all countries"
  ]
}
```
Extracted Arguments:
1. critical gaps persist
2. major shortfalls in priority development areas, such as gender equality (Goal 5), climate action (Goal 13), and peace, justice and strong institutions (Goal 16)
3. Approximately one third of indicators lack data for the past three years, hampering the ability of policymakers to make timely informed decisions and course corrections
4. Good trend data are a

## SGD 16: Peace, Justice, Strong Institutions

In [ ]:
topic = "SGD 16 (Peace, Justice, Strong Institutions): Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels"
sgd_number = "16"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
Model Output: ```json
{
  "arguments": [
    "Greatly increase funding to national and subnational governments and private businesses, especially in LICs and LMICs, to carry out needed SDG investments.",
    "Revise the credit rating system and debt sustainability metrics to facilitate long-term sustainable development.",
    "Revise liquidity structures for LICs and LMICs, especially regarding sovereign debts, to forestall self-fulfilling banking and balance-of-payments crises",
    "Create ambitious, internationally-agreed upon criteria for sustainable finance that are mandatory for all public financial institutions.",
    "Align private business investment flows with the SDGs, through improved national planning, regulation, reporting, and oversight.",
    "Reform current institutional frameworks and develop new mechanisms to improve the quality and speed of deployment of international cooperation, and monitor progress in an open and timely manner."
  ]
}
`

## SGD 17: Partnerships, sustainable development

In [ ]:
topic = "SGD 17 (Partnerships, sustainable development):Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development"
sgd_number = "17"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
Model Output: ```json
{
  "arguments": [
    "Greatly increase funding to national and subnational governments and private businesses, especially in LICs and LMICs, to carry out needed SDG investments.",
    "Revise the credit rating system and debt sustainability metrics to facilitate long-term sustainable development.",
    "Revise liquidity structures for LICs and LMICs, especially regarding sovereign debts, to forestall self-fulfilling banking and balance-of-payments crises;",
    "Create ambitious, internationally-agreed upon criteria for sustainable finance that are mandatory for all public financial institutions.",
    "Align private business investment flows with the SDGs, through improved national planning, regulation, reporting, and oversight.",
    "Reform current institutional frameworks and develop new mechanisms to improve the quality and speed of deployment of international cooperation, and monitor progress in an open and timely manner."
  ]
}


## SGD 0: Overarching terms

In [ ]:
topic = "SGD Overarching terms: Sustainable Development Goal, SDG, Agenda 2030, leave no one behind, Voluntary National Review, SDG transformations, "
sgd_number = "0"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
Model Output: ```json
{
  "arguments": [
    "At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.",
    "None of their objectives are beyond our reach.",
    "The SDGs are still achievable.",
    "It is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.",
    "To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.",
    "The Stimulus’ urgent objective is to address the chronic shortfall of international SDG financing confronting the LICs and LMICs, and to ramp up financing flows by at least US$500 billion by 2025.",
    "Greatly increase funding to national and subnational governments and private businesses, especially in LICs and LMICs, to carry out needed SDG investments.",
    "Revise the credit rating system and debt sustainability metrics to facilitate long-term sustainab

In [ ]:
!git add .
!git commit -m "sgd 2024 8-17"
!git push origin main  # or 'master' or your branch name

[main 21d563a] sgd 16, 17 0
 2 files changed, 35119 insertions(+), 1710 deletions(-)


error: src refspec # does not match any
error: src refspec or does not match any
error: src refspec 'master' does not match any
error: src refspec or does not match any
error: src refspec your does not match any
error: src refspec branch does not match any
error: src refspec name does not match any
error: failed to push some refs to 'https://github.com/camipalo/TFM.git'
